# Robustness check: cross-seed variance

Retrains each winning config (MLP and CNN) multiple times with different `training_seed` values, keeping `split_seed` fixed so every run trains/evaluates on the identical split. This measures how much the reported test accuracy varies from random init/shuffling alone, vs. being a single lucky run.

Same Kaggle setup as `kaggle_train.ipynb`/`kaggle_train_cnn.ipynb` — GPU + internet on, `mnist-mlp-pipeline` dataset (latest version, republished after this notebook's changes) and the `digit-recognizer` competition attached. Distribution-shift robustness is *not* run here — that part needs no retraining, so it was run locally against the already-downloaded checkpoints instead.

In [ ]:
!pip install -q pytorch-lightning torchmetrics optuna python-dotenv torchvision

In [ ]:
import sys

DATASET_DIR = '/kaggle/input/datasets/ivannamor/mnist-mlp-pipeline'
sys.path.insert(0, DATASET_DIR)
print('project code ready')

In [ ]:
from src.config import ExperimentConfig

mlp_config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/best_from_search.yaml')
mlp_config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'

cnn_config = ExperimentConfig.from_yaml(f'{DATASET_DIR}/configs/cnn_best_from_search.yaml')
cnn_config.data.data_dir = '/kaggle/input/competitions/digit-recognizer'

print(mlp_config)
print(cnn_config)

## Cross-seed trials -- MLP (5 seeds, ~5 full training runs)

In [ ]:
from src.robustness import run_cross_seed_trials

SEEDS = [42, 43, 44, 45, 46]
mlp_seed_results = run_cross_seed_trials(mlp_config, SEEDS)
for r in mlp_seed_results:
    print(r)

## Cross-seed trials -- CNN (5 seeds, ~5 full training runs)

In [ ]:
cnn_seed_results = run_cross_seed_trials(cnn_config, SEEDS)
for r in cnn_seed_results:
    print(r)

## Summary: mean +/- std across seeds

In [ ]:
import statistics

def summarize(name, results):
    accs = [r['test_accuracy'] for r in results]
    print(f"{name}: mean={statistics.mean(accs):.4f}  std={statistics.stdev(accs):.4f}  min={min(accs):.4f}  max={max(accs):.4f}")
    return accs

mlp_accs = summarize('MLP', mlp_seed_results)
cnn_accs = summarize('CNN', cnn_seed_results)